# 08 - Traitement des Données Images

## Section : Gestion des Images

## Objectif
Préparer les images pour la modélisation en :
- Validant et filtrant les images (détection des fichiers corrompus)
- Définissant les paramètres de prétraitement (taille, normalisation)
- Construisant le dataset final image (chemins + labels)
- Sauvegardant les métadonnées pour les notebooks suivants

## Plan
1. Chargement des données (exploration 07)
2. Validation des images (détection des corrompues)
3. Paramètres de prétraitement (224×224, normalisation ImageNet)
4. Construction du dataset final
5. Sauvegarde et statistiques

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.append(str(Path('../').resolve()))

from src.image import load_image_classification_data

ROOT = Path('../').resolve()
DATA_BRUT = ROOT / 'data brut'
IMAGE_DIR = ROOT / 'data' / 'processed' / 'image_train'
OUTPUT_DIR = ROOT / 'data' / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Configuration chargée")

## 1. Chargement et Construction du Dataset

In [ ]:
image_df = load_image_classification_data(DATA_BRUT, IMAGE_DIR, root=ROOT)
print(f"✅ {len(image_df):,} paires (image, label) chargées")
print(f"   Colonnes : {list(image_df.columns)}")
image_df.head()

## 2. Validation des Images (détection des corrompues)

In [ ]:
from PIL import Image

def is_valid_image(path: str, base_dir: Path, min_size: tuple = (10, 10)) -> bool:
    try:
        full_path = base_dir / path if not Path(path).is_absolute() else Path(path)
        with Image.open(full_path) as img:
            img.verify()
        with Image.open(full_path) as img:
            if img.width < min_size[0] or img.height < min_size[1]:
                return False
        return True
    except Exception:
        return False

print("🔄 Validation des images...")
valid_mask = [is_valid_image(p, ROOT) for p in image_df['image_path']]
n_valid = sum(valid_mask)
n_removed = len(image_df) - n_valid
if n_removed > 0:
    image_df = image_df[valid_mask].copy()
    print(f"   ⚠️ {n_removed} images corrompues exclues")
print(f"   ✅ {len(image_df):,} images valides")

## 3. Paramètres de Prétraitement

In [ ]:
PREPROCESS_CONFIG = {
    'target_size': (224, 224),
    'normalization': 'imagenet',
    'mean': [0.485, 0.456, 0.406],
    'std': [0.229, 0.224, 0.225],
}
print("📋 Paramètres de prétraitement (ResNet/ImageNet) :")
for k, v in PREPROCESS_CONFIG.items():
    print(f"   {k}: {v}")

## 4. Sauvegarde du Dataset

In [ ]:
output_path = OUTPUT_DIR / 'image_dataset_processed.csv'
image_df.to_csv(output_path, index=False)
print(f"✅ Dataset sauvegardé : {output_path}")
print(f"   - {len(image_df):,} lignes")
print(f"   - 27 classes")
print("\n✅ Prochaine étape : Notebook 09 - Modélisation baseline")